# Simulation d'un contrat de données partenaire
**Annexe facultative du bloc B5.** Aucun appel réseau. Le but est de tester des entrées et de conserver une décision explicite en cas de panne.

Scénario : la prévision logistique consomme une prévision météorologique. Une absence de pluie n'est pas identique à une absence de donnée. Les seuils ci-dessous sont des hypothèses pédagogiques.

In [1]:
from datetime import datetime, timezone, timedelta
MAINTENANT=datetime(2026,2,1,12,0,tzinfo=timezone.utc)
def verifier_message(message, maintenant=MAINTENANT):
    requis={"schema_version","zone","generated_at","rain_mm"}
    manquants=requis-set(message)
    if manquants:return False,"CHAMPS_MANQUANTS"
    if message["schema_version"]!="1.0":return False,"VERSION_INCOMPATIBLE"
    try:
        date=datetime.fromisoformat(message["generated_at"].replace("Z","+00:00"))
        if date.tzinfo is None:return False,"HORODATAGE_SANS_FUSEAU"
        pluie=float(message["rain_mm"])
    except (ValueError,TypeError):return False,"FORMAT_INVALIDE"
    if not 0<=pluie<=500:return False,"VALEUR_HORS_DOMAINE"
    age=maintenant-date
    if age<timedelta(0) or age>timedelta(hours=2):return False,"DONNEE_NON_FRAICHE"
    return True,"ACCEPTE"

messages=[
 {"schema_version":"1.0","zone":"Z1","generated_at":"2026-02-01T11:00:00Z","rain_mm":4.2},
 {"schema_version":"2.0","zone":"Z1","generated_at":"2026-02-01T11:00:00Z","rain_mm":4.2},
 {"schema_version":"1.0","zone":"Z1","generated_at":"2026-01-31T11:00:00Z","rain_mm":4.2},
 {"schema_version":"1.0","zone":"Z1","generated_at":"2026-02-01T11:00:00Z","rain_mm":-2}]
attendus=["ACCEPTE","VERSION_INCOMPATIBLE","DONNEE_NON_FRAICHE","VALEUR_HORS_DOMAINE"]
for message,attendu in zip(messages,attendus):
    ok,statut=verifier_message(message)
    assert statut==attendu
    decision="prévision enrichie" if ok else "prévision sans météo + alerte opérateur"
    print(statut,"→",decision)
print("4 tests de contrat réussis.")

ACCEPTE → prévision enrichie
VERSION_INCOMPATIBLE → prévision sans météo + alerte opérateur
DONNEE_NON_FRAICHE → prévision sans météo + alerte opérateur
VALEUR_HORS_DOMAINE → prévision sans météo + alerte opérateur
4 tests de contrat réussis.


## Analyse
Qui reçoit l'alerte ? Quel engagement de disponibilité demander au fournisseur ? Que journaliser sans conserver tout le contenu reçu ? Pourquoi ne pas remplacer la météo absente par zéro millimètre ?

**Corrigé.** Le propriétaire du service reçoit l'alerte selon sa criticité. Le délai de fraîcheur, les champs requis, la version et le mode dégradé doivent être convenus. Journaliser un identifiant de requête, le statut et la version avec une durée maîtrisée. Zéro signifie qu'aucune pluie n'est attendue ; une valeur absente signifie que l'on ne sait pas.

Cette simulation ne réalise pas l'authentification, le chiffrement, la gestion de quotas, une validation complète de schéma ou la mise en production d'une API. Ces couches restent obligatoires selon le contexte.